# Keycloak login and permissions
Why does the API answer `401` or `403`? Sign in the way the API's clients do, look inside the token (user, groups, roles), and check it against what each endpoint needs. Runs against the notebook's environment: your local stack, or a deployed one (dev, stage), which it only reads. Pick it on the Notebooks page, or call `target("dev")`.

On dev or stage it needs that environment's login in `.env.notebooks` (`BLUECORE_DEV_KEYCLOAK_USERNAME`, `…_PASSWORD`). Never type passwords into a notebook.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

## The realm
Keycloak's public OpenID configuration for the `bluecore` realm.

In [ ]:
realm = bc._s["keycloak_realm"]
oidc = bc.get(f"/keycloak/realms/{realm}/.well-known/openid-configuration")
print(oidc.status_code, oidc.url)
{k: oidc.json()[k] for k in ["issuer", "token_endpoint", "grant_types_supported"]} if oidc.ok else oidc.text[:300]

## Sign in and read the token
The token is a JWT: its middle part is plain JSON. This decodes it without checking the signature, which is fine for looking.

In [ ]:
import base64, json, datetime
from bluecore import NotConfigured

try:
    token = bc.keycloak_token()
except NotConfigured as err:
    token = None
    print(err)

def claims(jwt):
    part = jwt.split(".")[1]
    return json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))

if token:
    c = claims(token)
    when = lambda t: datetime.datetime.fromtimestamp(t, datetime.UTC).isoformat(timespec="seconds")
    print("user:", c.get("preferred_username"), "· subject:", c.get("sub"))
    print("client:", c.get("azp"), "· issued", when(c["iat"]), "· expires", when(c["exp"]))
    print("realm roles:", c.get("realm_access", {}).get("roles"))
    print("client roles:", {k: v.get("roles") for k, v in (c.get("resource_access") or {}).items()})
    print("groups:", c.get("groups"))

## What the API lets this user do
The API checks the token's roles: `create` (POST), `update` (PUT and DELETE) and `export`. `cataloger-read-only` blocks all of them.

In [ ]:
import pandas as pd

if token:
    roles = set(c.get("realm_access", {}).get("roles", [])) | {r for v in (c.get("resource_access") or {}).values() for r in v.get("roles", [])}
    read_only = "cataloger-read-only" in roles
    spec = bc.openapi()
    need = {"post": "create", "put": "update", "delete": "update"}
    rows = []
    for path, ops in spec["paths"].items():
        for m, op in ops.items():
            if op.get("security"):
                role = "export" if path == "/export/" else need.get(m, "create")
                rows.append({"method": m.upper(), "endpoint": path, "needs role": role, "allowed": role in roles and not read_only})
    allowed = pd.DataFrame(rows)
    print("Roles:", sorted(roles), "(read-only)" if read_only else "")
    display(allowed.groupby(["needs role", "allowed"]).size().rename("endpoints"))
    display(allowed[~allowed["allowed"]])

## Try it
The same safe request (an empty export, which the API refuses with `422` once you're allowed) without a token, with a broken one, and with yours: `401`/`403` means the login is the problem, `422` means it got through. Local stack only, as it's a POST.

In [ ]:
if token and not bc.read_only:
    for name, headers in [("no token", {}), ("broken token", {"Authorization": "Bearer nope"}), ("your token", {"Authorization": f"Bearer {token}"})]:
        r = bc.post("/api/export/", json={}, headers=headers)
        print(f"{name:<13} → {r.status_code} {r.text[:120]}")
else:
    print("Skipped: needs a token and your local stack.")